# 06.  LSTM Sentiment Analysis
Train an advanced bidirectional LSTM model with contextual attention on the enhanced dataset for 10 epochs.

In [12]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from collections import Counter
import re
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


In [13]:
# Load data - using the enhanced dataset generated recently
df = pd.read_csv('../data/processed/sentfin_strict.csv')

# Drop missing values if any
df = df.dropna(subset=['text', 'sentiment'])

# Encode sentiment labels
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
df['label'] = df['sentiment'].map(label_mapping)

print(f"Total samples: {len(df)}")
print("Sentiment distribution:")
print(df['sentiment'].value_counts())

# Train-test split (90% training, 10% testing)
train_df, test_df = train_test_split(df, test_size=0.10, random_state=42, stratify=df['label'])
print(f"Training samples: {len(train_df)}")
print(f"Testing samples: {len(test_df)}")

# For class weighting:
counts = train_df['label'].value_counts().sort_index()
class_counts = torch.tensor([counts.get(0, 0), counts.get(1, 0), counts.get(2, 0)], dtype=torch.float)
class_weights = (1.0 / class_counts)
class_weights = class_weights / class_weights.sum()
print(f"Computed Class Weights: {class_weights}")


Total samples: 9518
Sentiment distribution:
sentiment
neutral     3443
positive    3364
negative    2711
Name: count, dtype: int64
Training samples: 8566
Testing samples: 952
Computed Class Weights: tensor([0.3856, 0.3036, 0.3108])


In [14]:
def tokenize(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9]', ' ', text)
    return text.split()

# Build vocabulary
vocab = Counter()
for text in train_df['text']:
    vocab.update(tokenize(text))

# Remove rare words
min_freq = 2
vocab = {word: count for word, count in vocab.items() if count >= min_freq}

# Create word to index mapping
word2idx = {'<PAD>': 0, '<UNK>': 1}
for word in vocab:
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(word2idx)
print(f"Vocabulary size: {vocab_size}")

def encode_sentence(text, max_len=50):
    tokens = tokenize(text)
    encoded = [word2idx.get(word, word2idx['<UNK>']) for word in tokens]
    # mask = 1 for real tokens, 0 for pad
    mask = [1]*len(encoded)
    
    if len(encoded) < max_len:
        padding_len = max_len - len(encoded)
        encoded += [word2idx['<PAD>']] * padding_len
        mask += [0] * padding_len
    else:
        encoded = encoded[:max_len]
        mask = mask[:max_len]
    return encoded, mask

MAX_LEN = 50
encoded_train = [encode_sentence(text, MAX_LEN) for text in train_df['text']]
X_train = np.array([e[0] for e in encoded_train])
M_train = np.array([e[1] for e in encoded_train])
y_train = np.array(train_df['label'])

encoded_test = [encode_sentence(text, MAX_LEN) for text in test_df['text']]
X_test = np.array([e[0] for e in encoded_test])
M_test = np.array([e[1] for e in encoded_test])
y_test = np.array(test_df['label'])


Vocabulary size: 4768


In [15]:
class SentimentDataset(Dataset):
    def __init__(self, X, mask, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.mask = torch.tensor(mask, dtype=torch.bool)
        self.y = torch.tensor(y, dtype=torch.long)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.mask[idx], self.y[idx]

batch_size = 64
train_dataset = SentimentDataset(X_train, M_train, y_train)
test_dataset = SentimentDataset(X_test, M_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [16]:

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, 
                            hidden_dim, 
                            num_layers=n_layers, 
                            bidirectional=bidirectional, 
                            dropout=dropout,
                            batch_first=True)
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, text):
        # text: [batch size, seq len]
        embedded = self.dropout(self.embedding(text))
        
        # output: [batch size, seq len, hid dim * num directions]
        # hidden/cell: [num layers * num directions, batch size, hid dim]
        output, (hidden, cell) = self.lstm(embedded)
        
        if self.lstm.bidirectional:
            hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        else:
            hidden = self.dropout(hidden[-1,:,:])
            
        return self.fc(hidden)

# Hyperparameters
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
OUTPUT_DIM = 3 # Negative, Neutral, Positive
N_LAYERS = 2
BIDIRECTIONAL = True
DROPOUT = 0.5

model = LSTMClassifier(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, OUTPUT_DIM, N_LAYERS, BIDIRECTIONAL, DROPOUT)
model = model.to(device)
print(model)


LSTMClassifier(
  (embedding): Embedding(4768, 100, padding_idx=0)
  (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (fc): Linear(in_features=256, out_features=3, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [17]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

EPOCHS = 10
# Cosine annealing scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

def calculate_accuracy(preds, y):
    top_pred = preds.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]
    return acc

def train_step(model, batch_text, batch_mask, batch_labels):
    model.train()
    optimizer.zero_grad()
    # Fixed: Removed 'mask=batch_mask' to match your model's forward(self, text) definition
    predictions = model(batch_text) 
    loss = criterion(predictions, batch_labels)
    acc = calculate_accuracy(predictions, batch_labels)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    return loss.item(), acc.item()


In [18]:
train_losses = []
train_accs = []
best_loss = float('inf')

for epoch in range(EPOCHS):
    epoch_loss = 0
    epoch_acc = 0
    
    model.train() # Ensure model is in training mode
    
    for batch_X, batch_M, batch_y in train_loader:
        # Move data to GPU/CPU
        batch_X, batch_M, batch_y = batch_X.to(device), batch_M.to(device), batch_y.to(device)
        
        # Pass batch_X (text) and batch_M (mask) to your train_step
        # Ensure your train_step function is updated to handle these
        loss, acc = train_step(model, batch_X, batch_M, batch_y)
        
        epoch_loss += loss
        epoch_acc += acc
        
    # Update learning rate
    scheduler.step()
    
    avg_loss = epoch_loss / len(train_loader)
    avg_acc = epoch_acc / len(train_loader)
    
    # Save metrics
    train_losses.append(avg_loss)
    train_accs.append(avg_acc)
    
    # Track best loss
    if avg_loss < best_loss:
        best_loss = avg_loss
        # torch.save(model.state_dict(), 'best_model.pt') # Optional: save best model
    
    print(f'Epoch: {epoch+1:02} | Train Loss: {avg_loss:.3f} | Train Acc: {avg_acc*100:.2f}% | LR: {scheduler.get_last_lr()[0]:.6f}')


Epoch: 01 | Train Loss: 1.011 | Train Acc: 48.21% | LR: 0.000976
Epoch: 02 | Train Loss: 0.826 | Train Acc: 63.49% | LR: 0.000905
Epoch: 03 | Train Loss: 0.722 | Train Acc: 69.16% | LR: 0.000796
Epoch: 04 | Train Loss: 0.647 | Train Acc: 73.54% | LR: 0.000658
Epoch: 05 | Train Loss: 0.593 | Train Acc: 75.83% | LR: 0.000505
Epoch: 06 | Train Loss: 0.548 | Train Acc: 77.87% | LR: 0.000352
Epoch: 07 | Train Loss: 0.533 | Train Acc: 79.29% | LR: 0.000214
Epoch: 08 | Train Loss: 0.503 | Train Acc: 80.21% | LR: 0.000105
Epoch: 09 | Train Loss: 0.501 | Train Acc: 80.61% | LR: 0.000034
Epoch: 10 | Train Loss: 0.481 | Train Acc: 80.97% | LR: 0.000010


In [21]:
model.eval()
test_loss = 0
test_acc = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch_X, batch_M, batch_y in test_loader:
        batch_X, batch_M, batch_y = batch_X.to(device), batch_M.to(device), batch_y.to(device)
        predictions = model(batch_X)
        
        loss = criterion(predictions, batch_y)
        acc = calculate_accuracy(predictions, batch_y)
        
        test_loss += loss.item()
        test_acc += acc.item()
        
        all_preds.extend(predictions.argmax(1).cpu().numpy())
        all_labels.extend(batch_y.cpu().numpy())

avg_test_loss = test_loss / len(test_loader)
avg_test_acc = test_acc / len(test_loader)

print(f'Test Loss: {avg_test_loss:.3f} | Test Acc: {avg_test_acc*100:.2f}%')
print("\nClassification Report:")
target_names = ['Negative', 'Neutral', 'Positive']
print(classification_report(all_labels, all_preds, target_names=target_names))


Test Loss: 0.562 | Test Acc: 77.86%

Classification Report:
              precision    recall  f1-score   support

    Negative       0.81      0.75      0.78       271
     Neutral       0.74      0.80      0.77       344
    Positive       0.80      0.78      0.79       337

    accuracy                           0.78       952
   macro avg       0.78      0.78      0.78       952
weighted avg       0.78      0.78      0.78       952



In [20]:
torch.save(model.state_dict(), '../data/processed/1_lstm_sentiment_model.pt')
print("Improved model saved to ../data/processed/1_lstm_sentiment_model.pt")


Improved model saved to ../data/processed/1_lstm_sentiment_model.pt
